# Principio de Responsabilidad Única (SRP) — Cafetería

**Dominio propio:** sistema de pedidos de una cafetería.

El SRP dice que una clase debe tener **una sola razón para cambiar**. Aquí lo demuestro con la clase `Pedido`: primero la versión que mezcla varias responsabilidades (viola el SRP) y luego la versión que separa cada responsabilidad en su propia clase (cumple el SRP).

**Regla que aplico:** cada clase debe tener al menos 2 atributos y 2 métodos, y el ejemplo debe ejecutar sin errores.

## Versión que VIOLA el SRP

La clase `Pedido` hace de todo: guarda los datos del pedido, calcula el total, imprime el recibo y lo guarda en "base de datos". Tiene **tres razones para cambiar** (reglas de precio, formato del recibo y motor de persistencia).

In [2]:
class Pedido:
    def __init__(self, cliente: str, mesa: int) -> None:
        self.cliente: str = cliente
        self.mesa: int = mesa
        self.items: list[tuple[str, float]] = []

    def agregar_item(self, nombre: str, precio: float) -> None:
        self.items.append((nombre, precio))

    def calcular_total(self) -> float:
        return sum(precio for _, precio in self.items)

    def imprimir_recibo(self) -> None:
        # Responsabilidad de presentacion (no deberia vivir aqui)
        print(f"Recibo de {self.cliente} (mesa {self.mesa})")
        for nombre, precio in self.items:
            print(f"  - {nombre}: ${precio:.2f}")
        print(f"  TOTAL: ${self.calcular_total():.2f}")

    def guardar_en_db(self) -> None:
        # Responsabilidad de persistencia (no deberia vivir aqui)
        print(f"[DB] Guardando pedido de {self.cliente} con total ${self.calcular_total():.2f}")

In [3]:
# Ejemplo de uso (funciona, pero el diseno esta acoplado)
pedido = Pedido(cliente="Ana", mesa=5)
pedido.agregar_item("Latte", 8500)
pedido.agregar_item("Croissant", 4500)
pedido.imprimir_recibo()
pedido.guardar_en_db()

Recibo de Ana (mesa 5)
  - Latte: $8500.00
  - Croissant: $4500.00
  TOTAL: $13000.00
[DB] Guardando pedido de Ana con total $13000.00


### Problema

Si cambio el **formato del recibo**, si cambio el **motor de persistencia** o si cambian las **reglas de precio**, en los tres casos tengo que tocar la misma clase `Pedido`. Eso son tres razones para cambiar: viola el SRP y cada cambio arriesga romper lo demás.

## Versión que CUMPLE el SRP

Separo cada responsabilidad en su propia clase:

- `Pedido`: solo representa el pedido y su cálculo de total.
- `ReciboPrinter`: solo se encarga de la presentación.
- `PedidoRepository`: solo se encarga de la persistencia.

In [4]:
class Pedido:
    """Unica responsabilidad: representar el pedido y su total."""
    def __init__(self, cliente: str, mesa: int) -> None:
        self.cliente: str = cliente
        self.mesa: int = mesa
        self.items: list[tuple[str, float]] = []

    def agregar_item(self, nombre: str, precio: float) -> None:
        self.items.append((nombre, precio))

    def calcular_total(self) -> float:
        return sum(precio for _, precio in self.items)


class ReciboPrinter:
    """Unica responsabilidad: presentar el pedido como recibo."""
    def __init__(self, moneda: str = "$") -> None:
        self.moneda: str = moneda
        self.ancho: int = 32

    def formatear(self, pedido: Pedido) -> str:
        lineas = [f"Recibo de {pedido.cliente} (mesa {pedido.mesa})"]
        for nombre, precio in pedido.items:
            lineas.append(f"  - {nombre}: {self.moneda}{precio:.2f}")
        lineas.append(f"  TOTAL: {self.moneda}{pedido.calcular_total():.2f}")
        return "\n".join(lineas)

    def imprimir(self, pedido: Pedido) -> None:
        print(self.formatear(pedido))


class PedidoRepository:
    """Unica responsabilidad: persistir el pedido."""
    def __init__(self, nombre_tabla: str = "pedidos") -> None:
        self.nombre_tabla: str = nombre_tabla
        self.guardados: list[Pedido] = []

    def guardar(self, pedido: Pedido) -> None:
        self.guardados.append(pedido)
        print(f"[DB:{self.nombre_tabla}] Guardado pedido de {pedido.cliente} por {pedido.calcular_total():.2f}")

    def contar(self) -> int:
        return len(self.guardados)

In [5]:
# Ejemplo de uso: cada clase hace una sola cosa
pedido = Pedido(cliente="Ana", mesa=5)
pedido.agregar_item("Latte", 8500)
pedido.agregar_item("Croissant", 4500)

printer = ReciboPrinter(moneda="COP $")
repo = PedidoRepository()

printer.imprimir(pedido)
repo.guardar(pedido)
print("Pedidos guardados:", repo.contar())

Recibo de Ana (mesa 5)
  - Latte: COP $8500.00
  - Croissant: COP $4500.00
  TOTAL: COP $13000.00
[DB:pedidos] Guardado pedido de Ana por 13000.00
Pedidos guardados: 1


### Análisis

Ahora cada clase tiene **una única razón para cambiar**:

- Cambia el formato del recibo → solo toco `ReciboPrinter`.
- Cambia el motor de base de datos → solo toco `PedidoRepository`.
- Cambia la lógica de cálculo del pedido → solo toco `Pedido`.

El acoplamiento baja y cada pieza se puede probar y reutilizar de forma aislada.